In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import MessagePassing, global_mean_pool

from models_graph import HarmonicGraphEncoder
from models_FiLMatt import AttnFiLMSEModel

import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer

import pickle

import numpy as np

from graph_utils import get_random_bar_chords_from_data

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("data/gjt_train.pkl", "rb") as f:
    gjt_train = pickle.load(f)

In [3]:
# check the number of bars in each data entry
bar_counts = {}
for i, d in enumerate(gjt_train):
    num_bars = len(d['graph_ready_object'].bar_objects)
    if num_bars in bar_counts.keys():
        bar_counts[num_bars].append(i)
    else:
        bar_counts[num_bars] = [i]
print(bar_counts)

{16: [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 37, 38, 39, 40, 41, 43, 44, 45, 46, 47, 49, 50, 51, 53, 54, 55, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 72, 74, 75, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 118, 119, 120, 122, 123, 125, 126, 127, 129, 130, 131, 132, 133, 134, 135, 137, 138, 139, 140, 141, 142, 143, 144, 146, 148, 149, 150, 151, 152, 153, 154, 155, 156, 158, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 201, 202, 203, 204, 205, 206, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 221, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 238, 239, 241, 242, 244, 245, 246, 247, 248, 250, 251, 252, 254, 255, 256, 257

In [4]:
d = gjt_train[0]
print(d.keys())
d['graph_ready_object'].print_info()

dict_keys(['harmony_ids', 'attention_mask', 'pianoroll', 'time_signature', 'h_density_complexity', 'graph_ready_object'])
Number of bars: 16
No segment graph created yet.
Bar 1:
Bar token positions: [1, 2, 3, 4]
Number of chord objects in bar: 2
Chord object 1:
Chord label: A:min7
Pitch classes: [0, 4, 7, 9, 11]
Root: 9
Chord ID: 276
Bar Positions: [0, 1]
Token Positions: [1, 2]
Melody PCs: [array([], dtype=int64), array([ 9, 11])]
Graph Features:
tensor([[0., 1., 0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 1., 0., 1., 1.]])
Chord object 2:
Chord label: F#:hdim7
Pitch classes: [0, 4, 6, 9]
Root: 6
Chord ID: 194
Bar Positions: [2, 3]
Token Positions: [3, 4]
Melody PCs: [array([0]), array([0, 4])]
Graph Features:
tensor([[0., 0., 1., 0., 0., 1., 1., 0.],
        [0., 0., 0., 1., 0., 1., 1., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0.],
        [0., 1., 0., 

In [5]:
# get a random range of bars - at most 4
bars_range = np.random.randint(1, 5)
print(f'bars_range = {bars_range}')
bar_end = np.random.randint(bars_range, d['graph_ready_object'].num_bars + 1)
print(f'bar_end = {bar_end}')
bar_start = bar_end - bars_range
print(f'bar_start = {bar_start}')

bars_range = 4
bar_end = 16
bar_start = 12


In [6]:
d['graph_ready_object'].make_graph_of_segment(bar_start, bar_end)

In [7]:
data = d['graph_ready_object'].segment_graph
print(data)

HeteroData(
  pitch={ x=[12, 12] },
  event={ x=[7, 1] },
  (pitch, participates, event)={
    edge_index=[2, 36],
    edge_attr=[36, 8],
  },
  (event, next, event)={
    edge_index=[2, 6],
    edge_attr=[6, 7],
  }
)


In [8]:
graph_model = HarmonicGraphEncoder()

In [9]:
z = graph_model(data)

print(z.shape)

torch.Size([128])


In [10]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

model = AttnFiLMSEModel(
    chord_vocab_size=len(tokenizer.vocab),
    guidance_dim=graph_model.output_dim
)

In [11]:
model.freeze_base()
model.unfreeze_all()

In [12]:
melody_grid = torch.tensor([d['pianoroll']])
print(melody_grid.shape)
print(z.unsqueeze(0).shape)

torch.Size([1, 80, 13])
torch.Size([1, 128])


/tmp/ipykernel_3034047/2977237852.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:261.)
  melody_grid = torch.tensor([d['pianoroll']])


In [13]:
y, attn = model(melody_grid, z_g=z.unsqueeze(0), return_attn=True)

In [14]:
print(y.shape)
print(len(attn))
print(attn[0].keys())
print(attn[0]['pre_film_scores'].shape)
print(attn[0]['post_film_scores'].shape)
print(attn[0]['attention_probs'].shape)

torch.Size([1, 80, 355])
8
dict_keys(['pre_film_scores', 'post_film_scores', 'attention_probs'])
torch.Size([1, 8, 160, 160])
torch.Size([1, 8, 160, 160])
torch.Size([1, 8, 160, 160])


## Yes — your interpretation is mostly correct

Both `ParticipationMPNN` and `TemporalMPNN` are doing message passing: 
they compute edge-wise messages, aggregate them at destination nodes, 
and return updated node features.

But there are some important nuances:

### What they both do

1. `forward(...)` calls `self.propagate(...)`
2. `propagate(...)`:
   - prepares source/target features for each edge
   - calls `message(...)` for every edge
   - aggregates those messages per destination node using `aggr="add"`
   - calls `update(...)` if defined
   - returns the result

So yes, the output is a new node representation that encodes information 
flowing through the graph.

---

## Key difference between the two classes

### `ParticipationMPNN`
- Handles bipartite edges: `pitch -> event`
- Input is `x=(x_pitch, x_event)`
- `message(x_j, x_i, edge_attr)` computes a message from each pitch node 
to its event node
- `update(aggr_out, x)` combines:
  - the aggregated messages for each event node
  - the original event node features
- Result: updated event features that fuse pitch context and original 
event state

### `TemporalMPNN`
- Handles homogeneous edges: `event -> event`
- Input is a single `x` tensor
- `message(x_i, x_j, edge_attr)` computes messages between event nodes
- No custom `update(...)` is defined
- Default behavior: return the aggregated messages directly

So the key nuance is:
- `ParticipationMPNN` explicitly fuses old node state and incoming 
messages
- `TemporalMPNN` currently treats the aggregated messages as the final 
updated event state

---

## What “information flow” really means here

- `message(...)` computes how an edge transmits information
- `aggr="add"` combines all incoming edge messages for each destination node
- `update(...)` optionally refines the aggregated result using the existing 
node state
- The MLPs are the learnable functions that shape this process

So yes, the learned MLPs parameterize the flow. But the actual computation 
is not just “passing messages”; it is:
- building edge features from node pair + edge attrs,
- summarizing neighbors,
- optionally combining with the current node embedding,
- producing updated node embeddings.

---

## One more nuance

The output of `propagate()` is a tensor of updated node embeddings, not 
directly a task prediction. In your model:
- `ParticipationMPNN` produces updated event embeddings
- `TemporalMPNN` further refines those event embeddings
- then you pool and project to get the final graph vector `z`

That final `z` is what can be optimized for your task.

So your high-level understanding is good. The missing detail is that the 
graph neural layer is really about learning how to combine neighbor 
messages and node state, and `propagate()` orchestrates that automatically.